# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is sourced via a Croissant schema URL and contains structured clinicopathological data with multiple record sets and fields, uniquely referenced by their `@id`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and sample records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
# Access metadata
metadata = dataset.metadata

print('Dataset Name:', metadata.name)
print('Dataset Description:', metadata.description)
print('License:', metadata.license)
print('Version:', metadata.version)
print('Number of Record Sets:', len(metadata.recordSet))

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`). Each structured component and field is uniquely defined by its `@id`.

To explore the available record sets and their contents, enumerate them using the metadata. Then preview the first few records for each record set.

In [ ]:
# Gather all record set IDs from metadata
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if hasattr(rs, '@id'):
            record_sets.append(rs['@id'])
        elif hasattr(rs, '_id'):
            record_sets.append(rs._id)
        elif hasattr(rs, 'id'):
            record_sets.append(rs.id)
        elif hasattr(rs, 'id_'):
            record_sets.append(rs.id_)

# If record_sets is empty, attempt to fallback to a default record set ID from known schema structure
if not record_sets:
    # This dataset may have a single main record set
    # Use ID from the Croissant schema example
    main_record_set_id = 'https://api.app.sen.science/frontiers/7862866/629c16ec-37ee-4556-a351-d5164116c2dd#recordSet'
    record_sets = [main_record_set_id]
else:
    main_record_set_id = record_sets[0]

print('Available Record Sets (`@id`):')
for rs_id in record_sets:
    print('-', rs_id)

# Preview records from the main record set
print(f'Preview records from {main_record_set_id}:')
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if i >= 2:
        break

### Listing Fields and Columns by `@id`
In Croissant, each field (variable) and column is uniquely referenced by its `@id`. We will enumerate and show the available fields for the main record set.

In [ ]:
# List fields from the main record set
main_rs = None
for rs in getattr(metadata, 'recordSet', []):
    if hasattr(rs, '@id') and rs['@id'] == main_record_set_id:
        main_rs = rs
    elif hasattr(rs, '_id') and rs._id == main_record_set_id:
        main_rs = rs
    elif hasattr(rs, 'id') and rs.id == main_record_set_id:
        main_rs = rs
    elif hasattr(rs, 'id_') and rs.id_ == main_record_set_id:
        main_rs = rs

if not main_rs:
    print('Unable to access structured fields for the main record set.')
else:
    field_ids = []
    if hasattr(main_rs, 'field') and main_rs.field:
        for fld in main_rs.field:
            if hasattr(fld, '@id'):
                field_ids.append(fld['@id'])
            elif hasattr(fld, '_id'):
                field_ids.append(fld._id)

    print('Fields (Variables) `@id` in the Record Set:')
    for fid in field_ids:
        print('-', fid)

## 3. Data Extraction
Load all data from the main record set into a DataFrame for further analysis. All column names are referenced by their `@id` from the metadata field list.

In [ ]:
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print columns for the main record set referenced by its `@id`
print(f"Columns for {main_record_set_id} (referenced by `@id`):")
print(dataframes[main_record_set_id].columns.tolist())
# Show first few records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering by numeric field, normalizing, and grouping using strictly the field `@id`.

In [ ]:
# Choose numeric and grouping fields by their `@id` for demo
# Let's assume '@id:Age' and '@id:Sex' are present as per description
numeric_field_id = 'Age'
group_field_id = 'Sex'

# Ensure columns (may be different, list columns again)
print('Frame columns:', dataframes[main_record_set_id].columns.tolist())

# If 'Age' and 'Sex' aren't available, use another numeric/groupable column from the dataset
if numeric_field_id not in dataframes[main_record_set_id].columns:
    # Try with actual field ids if provided
    numeric_field_id = dataframes[main_record_set_id].select_dtypes(['number']).columns[0]
    print(f"Numeric field selected: {numeric_field_id}")
if group_field_id not in dataframes[main_record_set_id].columns:
    for col in dataframes[main_record_set_id].columns:
        if dataframes[main_record_set_id][col].dtype == 'object' and col.lower().find('sex') >= 0:
            group_field_id = col
            break

threshold = 60
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id
if group_field_id in dataframes[main_record_set_id].columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and the relationship to the grouping field. All references must use the column `@id` from the record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(dataframes[main_record_set_id][numeric_field_id], bins=10, kde=True)
plt.title(f'Histogram of {numeric_field_id} (by `@id`)')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_record_set_id])
    plt.title(f'{numeric_field_id} distribution by {group_field_id} (referenced by `@id`)')
    plt.show()

## 6. Conclusion
This notebook demonstrates loading, exploring, and visualizing the FAIR² clinical dataset using `mlcroissant`.

- Data can be programmatically explored using Croissant schemas and strict `@id` references.
- Key variables, e.g., Age and Sex, allow stratification and normalization for analytic pipelines.
- Filtering, grouping, and visualization can be seamlessly applied to medical datasets for further research.

For more advanced analytics or applications, consult the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) or the FAIR² schema for additional fields and record set relationships.